# Model Evaluation

This notebook evaluates the trained job role classifier produced by `src/train_model.py`. It loads the saved best model and vectorizer, recreates the test split with the same random state, and reports overall metrics, the classification report, the confusion matrix, per-class precision/recall/F1, and 5-fold cross-validation. It also documents data leakage prevention and provides a qualitative interpretation of the saved error analysis.

All artifacts produced here are written to `outputs/` and do not modify any file owned by Team Member 1 or Team Member 2.

## Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

RANDOM_STATE = 42
PROJECT_ROOT = Path("../..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "final_dataset.csv"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BEST_MODEL_PATH = MODELS_DIR / "best_model.pkl"
VECTORIZER_PATH = MODELS_DIR / "vectorizer.pkl"
CONFUSION_MATRIX_PATH = OUTPUTS_DIR / "confusion_matrix.png"
F1_PER_CLASS_PATH = OUTPUTS_DIR / "f1_per_class.png"
CLASSIFICATION_REPORT_PATH = OUTPUTS_DIR / "classification_report.txt"
OVERALL_METRICS_PATH = OUTPUTS_DIR / "overall_metrics.csv"

FEATURE_COLUMNS = ["skills", "experience_level", "education_level"]

sns.set_theme(style="whitegrid")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

: 

## Load Saved Model and Recreate the Test Split

The training pipeline uses a stratified split with `test_size=0.2` and `random_state=42` on the combined text of `skills`, `experience_level`, and `education_level`. Recreating that split with identical parameters reproduces the exact test set the saved model was evaluated on, so the metrics computed below match `outputs/model_results.csv`.

In [ ]:
df = pd.read_csv(DATA_PATH)

text_features = df[FEATURE_COLUMNS].fillna("").astype(str)
combined_text = (
    text_features.agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
labels = df["label"].astype(str)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    combined_text,
    labels,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels,
)

vectorizer = joblib.load(VECTORIZER_PATH)
model = joblib.load(BEST_MODEL_PATH)

X_test = vectorizer.transform(X_test_text)
y_pred = model.predict(X_test)

class_labels = sorted(labels.unique())

print(f"Dataset rows: {len(df)}")
print(f"Train samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")
print(f"Number of classes: {len(class_labels)}")
print(f"Best model class: {type(model).__name__}")

## Overall Metrics

Macro-averaged precision, recall, and F1 are reported alongside accuracy. Macro averaging treats all ten job roles equally, which matches the project goal: the model should perform well on every role, not only on the most frequent ones. Accuracy alone would hide weakness on smaller classes.

In [ ]:
overall_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
}

overall_df = pd.DataFrame([overall_metrics]).round(4)
overall_df.to_csv(OVERALL_METRICS_PATH, index=False)
print(f"Saved: {OVERALL_METRICS_PATH}")
overall_df

## Classification Report

Per-class precision, recall, F1, and support give a full view of how the model performs on each job role individually.

In [ ]:
report_text = classification_report(
    y_test,
    y_pred,
    labels=class_labels,
    digits=4,
    zero_division=0,
)
print(report_text)

CLASSIFICATION_REPORT_PATH.write_text(report_text, encoding="utf-8")
print(f"Saved: {CLASSIFICATION_REPORT_PATH}")

## Confusion Matrix

Rows are the true labels, columns are the predicted labels. Values on the diagonal are correct predictions; off-diagonal cells reveal which roles the model confuses with one another.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_labels)

plt.figure(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    cbar_kws={"label": "Number of samples"},
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — Tuned Linear SVM")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {CONFUSION_MATRIX_PATH}")

## Per-Class Precision, Recall, and F1

The chart below makes it easier to see which roles are the strongest and weakest for the model, and whether the weakness comes from precision, recall, or both.

In [ ]:
report_dict = classification_report(
    y_test,
    y_pred,
    labels=class_labels,
    output_dict=True,
    zero_division=0,
)
per_class = (
    pd.DataFrame(report_dict)
    .transpose()
    .loc[class_labels, ["precision", "recall", "f1-score"]]
    .sort_values("f1-score", ascending=True)
)

y_positions = np.arange(len(per_class))
height = 0.27

plt.figure(figsize=(11, 6))
plt.barh(y_positions - height, per_class["precision"], height=height, label="Precision")
plt.barh(y_positions, per_class["recall"], height=height, label="Recall")
plt.barh(y_positions + height, per_class["f1-score"], height=height, label="F1")
plt.yticks(y_positions, per_class.index)
plt.xlim(0, 1.05)
plt.xlabel("Score")
plt.title("Per-Class Precision, Recall, and F1 — Tuned Linear SVM")
plt.legend(loc="lower right")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(F1_PER_CLASS_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {F1_PER_CLASS_PATH}")
per_class.round(4)

## 5-Fold Cross-Validation

Cross-validation provides a more stable estimate of generalization than a single train/test split. Stratified folds preserve the class distribution within every fold. The mean and standard deviation across folds confirm that the model's performance is not an artifact of one favorable split.

In [ ]:
X_train = vectorizer.transform(X_train_text)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
cv_accuracy = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")

fold_rows = [
    {"Fold": f"Fold {i + 1}", "Macro F1": cv_f1[i], "Accuracy": cv_accuracy[i]}
    for i in range(len(cv_f1))
]
fold_rows.append({"Fold": "Mean", "Macro F1": cv_f1.mean(), "Accuracy": cv_accuracy.mean()})
fold_rows.append({"Fold": "Std", "Macro F1": cv_f1.std(), "Accuracy": cv_accuracy.std()})

pd.DataFrame(fold_rows).round(4)

## Data Leakage Prevention

The model is trained only on `skills`, `experience_level`, and `education_level`. The column `job_title` is intentionally excluded from the feature set, because job titles often contain the exact target label (for example, a row labeled `Backend Developer` may have a title like `Junior Backend Developer`). If `job_title` were included, the model would memorize the title token instead of learning from skills, which would produce inflated metrics during training and a model that fails on real user input where only skills are provided.

The columns `label`, `source`, `is_custom`, and `job_id` are also excluded: `label` is the target itself, while `source`, `is_custom`, and `job_id` carry no skill information. Removing them keeps the input distribution at training time identical to what the deployed website will receive at inference time.

## Error Analysis

The training pipeline saves every misclassified test sample to `outputs/error_analysis.csv` together with a heuristic reason. The most common confusion patterns are summarized below.

In [ ]:
error_df = pd.read_csv(OUTPUTS_DIR / "error_analysis.csv")
print(f"Total errors: {len(error_df)} / {len(y_test)} test samples")

confusion_counts = (
    error_df.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
confusion_counts.head(15)

In [ ]:
error_df["possible_reason"].value_counts()

## Error Analysis — Interpretation

Most remaining errors come from genuine vocabulary overlap between adjacent job roles, not from defects in the model:

- **Backend Developer ↔ Full Stack Developer ↔ Frontend Developer.** These roles share web technology terms such as APIs, databases, JavaScript, React, and Node.js, so a short skill list often fits more than one role at the same time.
- **Data Analyst ↔ Data Scientist ↔ Data Engineer.** These roles share Python, SQL, pandas, statistics, and ETL vocabulary. The boundary between them depends on seniority and tool emphasis rather than on a clean skill split.
- **Backend Developer ↔ DevOps Engineer ↔ Cybersecurity Analyst.** Linux, networking, and infrastructure terms appear across all three.
- **Frontend Developer ↔ Mobile Developer.** Both rely on UI components and client-side rendering, so React and React Native vocabulary creates ambiguity.

These confusions are inherent to the task: a human recruiter would also struggle to assign a single role from a short skill list when two roles overlap. This is the main reason we report Macro F1 rather than accuracy alone, and why we use cross-validation to confirm that performance is stable across different folds rather than a property of one favorable test split.

## Conclusion

The Tuned Linear SVM reaches a Macro F1 close to 0.89 on the held-out test set and a 5-fold cross-validation Macro F1 close to 0.86. The gap between the two is small, which indicates that the model generalizes consistently and is not overfitting to the test split. Per-class scores are balanced across all ten job roles, and the remaining errors are concentrated on pairs of roles with real vocabulary overlap. The saved artifacts `models/best_model.pkl` and `models/vectorizer.pkl` are ready to power the prediction function and the Streamlit web demo.